In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

torch.cuda.is_available()

True

In [3]:
from next_sequence.core import TaskType, Feature, CoreDataset
from next_sequence.collate_fn import CustomCollate
from next_sequence.utils import create_bow
from next_sequence.embedding import EmbeddingModel
from next_sequence.fusion import BaseFusion
from next_sequence.create_cls import PrependCLS
from next_sequence.attention_head import MultiHeadAttention
from next_sequence.position_embedding import RoPE, RoPEStrategy
from next_sequence.encoder_block import EncoderBlock, EncoderCLSBackbone
from typing import Any
from next_sequence.multi_task_head import MultiTaskHeads
from next_sequence.multi_task_model import MultiTaskModel
import torch.nn as nn

# Constant

In [4]:
D_MODEL = 512
N_HEADS = 2

# Mock Dataset

In [5]:
import pandas as pd

df = pd.DataFrame({
    'id': [1,2,3],
    'item': [['a','a','b'], ['a','b'], ['a','a','a','a','b']],
    'age': [[25,26,27], [35,37], [41,41,42,42,55]],
    'price': [[100, 110, 120], [1000, 1500], [1000, 1000, 1000, 1000, 2500]],
})

df.head()

,id,item,age,price
0,1,"[a, a, b]","[25, 26, 27]","[100, 110, 120]"
1,2,"[a, b]","[35, 37]","[1000, 1500]"
2,3,"[a, a, a, a, b]","[41, 41, 42, 42, 55]","[1000, 1000, 1000, 1000, 2500]"


In [6]:
item_bow, item_vocab_size, max_len = create_bow(df['item'].tolist())
df['item'] = df['item'].apply(lambda x: [item_bow[_x] for _x in x])
df

,id,item,age,price
0,1,"[2, 2, 1]","[25, 26, 27]","[100, 110, 120]"
1,2,"[2, 1]","[35, 37]","[1000, 1500]"
2,3,"[2, 2, 2, 2, 1]","[41, 41, 42, 42, 55]","[1000, 1000, 1000, 1000, 2500]"


# Create batch

In [7]:
samples = df.to_dict(orient='records')
dataset = CoreDataset(samples)
batch = dataset[:]
batch

[{'id': 1, 'item': [2, 2, 1], 'age': [25, 26, 27], 'price': [100, 110, 120]},
 {'id': 2, 'item': [2, 1], 'age': [35, 37], 'price': [1000, 1500]},
 {'id': 3,
  'item': [2, 2, 2, 2, 1],
  'age': [41, 41, 42, 42, 55],
  'price': [1000, 1000, 1000, 1000, 2500]}]

# Custom Collate (you can apply to your case)

In [8]:
# all custom collate_fn here will be changed project by project
def _pack_xy(b, dtype):
    return torch.tensor(b, dtype=dtype)

def collate_common(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.long) for b in batch], None
    
    x = [_pack_xy(b[:-1], torch.long) for b in batch]
    y = [_pack_xy(b[-1], torch.long) for b in batch]
    return x, y

def collate_delta(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.float) for b in batch], None

    x = [_pack_xy(b[:-1], torch.float) for b in batch]
    y = [_pack_xy(b[-1]-b[-2], torch.float) for b in batch]
    return x, y

def collate_delta_bin(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.long) for b in batch], None

    def logic(b):
        if b <= 1:
            return 1
        if b <= 3:
            return 2
        return 3
    # if embedding is linear, x dtype must be float only
    x = [_pack_xy(b[:-1], torch.float) for b in batch]
    y = [_pack_xy(logic(b[-1]-b[-2]), torch.long) for b in batch]
    return x, y

In [9]:
features = [
    Feature(name='item', task=TaskType.CLASSIFICATION, vocab_size=item_vocab_size),
    Feature(name='age', task=TaskType.CLASSIFICATION, vocab_size=3+1, is_linear=True),
    Feature(name='price', task=TaskType.REGRESSION, is_linear=True),
]
funcs = {
    'item': collate_common,
    'age': collate_delta_bin,
    'price': collate_delta
}
collate_fn = CustomCollate(features, 'item', funcs, inference=False)

In [10]:
prep_data = collate_fn(batch)
prep_data

{'X': defaultdict(list,
             {'item': tensor([[2, 2, 0, 0],
                      [2, 0, 0, 0],
                      [2, 2, 2, 2]]),
              'age': tensor([[[25.],
                       [26.],
                       [ 0.],
                       [ 0.]],
              
                      [[35.],
                       [ 0.],
                       [ 0.],
                       [ 0.]],
              
                      [[41.],
                       [41.],
                       [42.],
                       [42.]]]),
              'price': tensor([[[ 100.],
                       [ 110.],
                       [   0.],
                       [   0.]],
              
                      [[1000.],
                       [   0.],
                       [   0.],
                       [   0.]],
              
                      [[1000.],
                       [1000.],
                       [1000.],
                       [1000.]]])}),
 'Y': defaultdict(list,
  

# Embedding

In [11]:
embedder = EmbeddingModel(features=features, d_model=D_MODEL)

In [12]:
prep_data['X']['item']

tensor([[2, 2, 0, 0],
        [2, 0, 0, 0],
        [2, 2, 2, 2]])

In [13]:
vectors = embedder(prep_data['X'])
for feature in features:
    print(vectors[feature.name].shape)

torch.Size([3, 4, 512])
torch.Size([3, 4, 512])
torch.Size([3, 4, 512])


# Fusion Strategy (you can apply to your case)

In [14]:
class SumFusion(BaseFusion):
    def forward(self, X:dict[str, Any])->Any:
        return torch.stack([
            X[feature.name]
            for feature in self.features
        ]).sum(0)

    def output_dim(self)->int:
        return self.d_model

class ConcatFusion(BaseFusion):
    def forward(self, X:dict[str, Any])->Any:
        return torch.cat([X[feature.name] for feature in self.features], dim=-1)
    def output_dim(self)->int:
        return len(self.features) * self.d_model

class ConcatProjectionFusion(BaseFusion):
    def __init__(self, features, d_model, d_out=None):
        self._target = d_out or d_model
        super().__init__(features, d_model)
        self.proj = nn.Linear(len(features) * d_model, self._target)
    def forward(self, X):
        return self.proj(torch.cat([X[f.name] for f in self.features], dim=-1))
    def output_dim(self):
        return self._target

In [15]:
fusion = SumFusion(features=features, d_model=D_MODEL)
fusion.d_out, fusion(vectors).shape

(512, torch.Size([3, 4, 512]))

In [16]:
fusion = ConcatFusion(features=features, d_model=D_MODEL)
fusion.d_out, fusion(vectors).shape

(1536, torch.Size([3, 4, 1536]))

In [17]:
fusion = ConcatProjectionFusion(features=features, d_model=D_MODEL, d_out=int(D_MODEL/2))
fusion.d_out, fusion(vectors).shape

(256, torch.Size([3, 4, 256]))

In [18]:
prep_data['mask']

tensor([[False, False,  True,  True],
        [False,  True,  True,  True],
        [False, False, False, False]])

In [19]:
fusion = SumFusion(features=features, d_model=D_MODEL)
fused_vectors = fusion(vectors)
fused_vectors.shape

torch.Size([3, 4, 512])

# Prepend CLS which will be used later in the multi-task prediction

In [20]:
prepend_cls = PrependCLS(d_model=fusion.d_model)
x, mask = prepend_cls(fused_vectors, prep_data['mask'])

In [21]:
x.shape, mask.shape

(torch.Size([3, 5, 512]), torch.Size([3, 5]))

In [22]:
mask

tensor([[False, False, False,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False, False, False]])

# Attention Head with RoPE

In [23]:
rope = RoPE(fusion.d_out // N_HEADS, max_len, RoPEStrategy.INTERLEAVED)

In [24]:
attn = MultiHeadAttention(d_model=fusion.d_model, n_heads=N_HEADS, rope=rope)

In [25]:
out = attn(x, mask)
out.shape

torch.Size([3, 5, 512])

In [26]:
x.shape, out.shape, mask.shape

(torch.Size([3, 5, 512]), torch.Size([3, 5, 512]), torch.Size([3, 5]))

# Test EncoderCLSBlock (combine, rope, attn together)

In [27]:
block = EncoderBlock(d_model=fusion.d_out, n_heads=N_HEADS, rope=rope, expansion=4, dropout=0.1)

In [28]:
b_x = block(x, mask)
x.shape, b_x.shape, mask.shape

(torch.Size([3, 5, 512]), torch.Size([3, 5, 512]), torch.Size([3, 5]))

In [90]:
emb = EmbeddingModel(features, D_MODEL)
fusion = SumFusion(features, D_MODEL)
prepend_cls = PrependCLS(fusion.d_out)
rope = RoPE(fusion.d_out // N_HEADS, max_len + 1)
backbone = EncoderCLSBackbone(emb, fusion, prepend_cls, rope, N_HEADS)

In [91]:
backbone.eval()
with torch.no_grad():
    representation = backbone(prep_data['X'], prep_data['mask'])
representation.shape

torch.Size([3, 512])

In [92]:
representation

tensor([[ 0.8622,  0.2062,  0.4727,  ..., -1.7516,  0.8717, -0.3531],
        [ 1.0018,  0.1041,  0.2251,  ..., -1.1875,  0.6646, -0.5299],
        [ 0.6485, -0.2659,  0.7175,  ..., -1.3467,  0.5500, -0.0138]])

# Predict with MultiTaskHeads

In [93]:
mlh = MultiTaskHeads(features=features, d_model=fusion.d_out)

In [94]:
mlh.eval()
with torch.no_grad():
    predictions = mlh(representation)

In [95]:
predictions.keys()

dict_keys(['item', 'age', 'price'])

In [96]:
predictions['item'].softmax(dim=-1)

tensor([[0.2294, 0.3186, 0.4519],
        [0.2292, 0.4070, 0.3638],
        [0.2356, 0.4352, 0.3293]])

In [97]:
predictions['age'].softmax(dim=-1)

tensor([[0.3533, 0.0857, 0.1469, 0.4141],
        [0.4395, 0.0895, 0.1219, 0.3491],
        [0.3163, 0.0713, 0.1619, 0.4505]])

In [98]:
predictions['price']

tensor([[-0.6304],
        [-0.5608],
        [-0.7978]])

# Model combined backbone, mlh

In [99]:
model = MultiTaskModel(backbone, mlh)

In [100]:
model.eval()
with torch.no_grad():
    _predictions = model(prep_data['X'], prep_data['mask'])

In [101]:
_predictions['item'].softmax(dim=-1)

tensor([[0.2294, 0.3186, 0.4519],
        [0.2292, 0.4070, 0.3638],
        [0.2356, 0.4352, 0.3293]])

In [102]:
_predictions['age'].softmax(dim=-1)

tensor([[0.3533, 0.0857, 0.1469, 0.4141],
        [0.4395, 0.0895, 0.1219, 0.3491],
        [0.3163, 0.0713, 0.1619, 0.4505]])

In [103]:
_predictions['price']

tensor([[-0.6304],
        [-0.5608],
        [-0.7978]])

In [104]:
model.eval()
with torch.no_grad():
    rep = backbone(prep_data['X'], prep_data['mask'])
    manual = mlh(rep)
    wrapped = model(prep_data['X'], prep_data['mask'])

all(torch.allclose(manual[k], wrapped[k], atol=1e-6) for k in manual)   # expect True

True